In [1]:
!pip install -q ragas datasets accelerate bitsandbytes
!pip install -q transformers sentencepiece
!pip install -q langchain langchain-community langchain-huggingface
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.7/360.7 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently tak

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
GG_COLAB = "/content/drive/MyDrive"
LABORLAW_DIR = f'{GG_COLAB}/laborlaw/'


In [5]:
import json
import csv
import os
import time
import sys
import random
from tqdm import tqdm
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas import RunConfig

from ragas.metrics import (AnswerRelevancy,
    # Faithfulness,
    # ContextPrecision,
    # ContextRecall,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import HuggingFaceEmbeddings
from transformers import (AutoTokenizer,AutoModelForCausalLM, pipeline,)
import torch
from langchain_huggingface import (HuggingFacePipeline,HuggingFaceEmbeddings,)
GTRUTH_FILE   = f'{LABORLAW_DIR}gtruth_qa.json'
RAG_FILE      = f'{LABORLAW_DIR}rag_qa.json'
GRAPHRAG_FILE = f'{LABORLAW_DIR}graphrag_qa.json'
LLMONLY_FILE  = f'{LABORLAW_DIR}qwen_qa.json'
CHECKPOINT_FILE = f'{LABORLAW_DIR}ragas/ragas_checkpoint.json'
PARTIAL_CSV  = f'{LABORLAW_DIR}ragas/partial_results.csv'
SUMMARY_JSON = f'{LABORLAW_DIR}ragas/results_summary.json'
DETAIL_CSV   = f'{LABORLAW_DIR}ragas/results_detail.csv'

JUDGE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
# JUDGE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
run_config = RunConfig(
    max_workers=1,
    timeout=300,
    max_retries=3,
)

BATCH_SIZE = 2
METRIC_KEYS = [
    "answer_relevancy",
    "faithfulness",
    "context_precision",
    "context_recall",
]

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

print("Loading model...")
from transformers import BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)
model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL,
    device_map="auto",
    dtype=torch.float16,
    quantization_config=bnb_config,
)

# pipe = pipeline(
#     "text-generation",
#     model=model,
#     tokenizer=tokenizer,
#     max_new_tokens=64,
#     temperature=0,
#     do_sample=False,
# )

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id,

)
hf_llm = HuggingFacePipeline(pipeline=pipe)
llm = LangchainLLMWrapper(hf_llm)
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL,)

metrics = [
    AnswerRelevancy(),
    # Faithfulness(),
    # ContextPrecision(),
    # ContextRecall(),
]

def read_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def load_checkpoint():
    if not os.path.exists(CHECKPOINT_FILE):
        return None

    with open(CHECKPOINT_FILE, encoding="utf-8") as f:
        return json.load(f)

def save_checkpoint(pipeline_name, next_index):
    with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
        json.dump(
            {
                "pipeline": pipeline_name,
                "next_index": next_index,
            },
            f,
        )

def clear_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        os.remove(CHECKPOINT_FILE)

def append_to_partial_csv(df_batch):
    write_header = not os.path.exists(PARTIAL_CSV)

    df_batch.to_csv(
        PARTIAL_CSV,
        mode="a",
        header=write_header,
        index=False,
        encoding="utf-8-sig",
    )

def format_eta(seconds):
    minutes = int(seconds // 60)
    secs = int(seconds % 60)

    return f"{minutes}p {secs}s"

ground_truth_list = read_json(GTRUTH_FILE)
gtruth_map = {}
for item in ground_truth_list:
    gtruth_map[item["id"]] = item

def build_rows(path, pipeline_name):
    rows = []
    for d in read_json(path):
        if d.get("id") not in gtruth_map:
            continue
        if pipeline_name == "LLM_only":
            contexts = []
        else:
            ctx = d.get("context_text") or ""
            contexts = [ctx] if ctx else []
        rows.append(
            {
                "id": d["id"],
                "question": d["question"],
                "answer": d["answer"],
                "contexts": contexts,
                "ground_truth": gtruth_map[d["id"]]["answer"],
                "question_type": d.get(
                    "question_type",
                    "unknown",
                ),
            }
        )

    return rows
# cac pipeline
pipelines = {
    "RAG": build_rows(RAG_FILE, "RAG"),
    "GraphRAG": build_rows(
        GRAPHRAG_FILE,
        "GraphRAG",
    ),
    "LLM_only": build_rows(
        LLMONLY_FILE,
        "LLM_only",
    ),
}

print("\nSố câu hỏi:")

for name, rows in pipelines.items():
    print(f"{name}: {len(rows)}")
# eval batch
def eval_batch(batch_rows, pipeline_name, batch_index):
    dataset = Dataset.from_list(
        [
            {
                "question": r["question"],
                "answer": r["answer"],
                "contexts": r["contexts"],
                "ground_truth": r["ground_truth"],
            }
            for r in batch_rows
        ]
    )
    try:
        result = evaluate(
            dataset,
            metrics=metrics,
            llm=llm,
            embeddings=embeddings,
            run_config=run_config,
            raise_exceptions=False,
        )
        return result
    except Exception as e:
        print(f"\nERROR batch {batch_index}: {e}")
        return None

# main loop
checkpoint = load_checkpoint()
all_dfs = {}
pipeline_names = list(pipelines.keys())
for pipeline_name in pipeline_names:
    rows = pipelines[pipeline_name]
    start_index = 0
    if checkpoint and checkpoint["pipeline"] == pipeline_name:
        start_index = checkpoint["next_index"]
    print("\n=================================")
    print(f"Pipeline: {pipeline_name}" )
    print("=================================")
    pipeline_dfs = []
    time_per_batch = []
    for start in tqdm(range(start_index, len(rows), BATCH_SIZE),desc=pipeline_name,):
        batch_rows = rows[start:start + BATCH_SIZE]
        batch_index = start // BATCH_SIZE
        t0 = time.time()
        result = eval_batch(batch_rows,pipeline_name,batch_index,)
        if result is None:
            save_checkpoint(pipeline_name,start + BATCH_SIZE, )
            continue
        df_batch = result.to_pandas()
        df_batch["question_type"] = [
            r["question_type"]
            for r in batch_rows
        ]
        df_batch["id"] = [
            r["id"]
            for r in batch_rows
        ]

        df_batch["pipeline"] = pipeline_name
        pipeline_dfs.append(df_batch)
        append_to_partial_csv(df_batch)
        elapsed = time.time() - t0
        time_per_batch.append(elapsed)
        avg_time = ( sum(time_per_batch)  / len(time_per_batch))
        remaining_batches = ( len(rows) - (start + BATCH_SIZE)) / BATCH_SIZE
        eta_seconds = avg_time * max( remaining_batches, 0,)
        tqdm.write(
            f"Batch {batch_index}"
            f" | {elapsed:.1f}s"
            f" | ETA {format_eta(eta_seconds)}"
        )
        save_checkpoint(pipeline_name,start + BATCH_SIZE,)
    if pipeline_dfs:
        all_dfs[pipeline_name] = pd.concat(pipeline_dfs,ignore_index=True,)
# save
clear_checkpoint()
if not all_dfs:
    print("Không có kết quả.")
    sys.exit(1)
summary = {}
for name, df in all_dfs.items():
    summary[name] = {}
    for k in METRIC_KEYS:
        if k in df.columns:
            summary[name][k] = round(float(df[k].mean()), 4, )

print("\n========================")
print("SUMMARY")
print("========================")

for name, vals in summary.items():
    print(f"\n{name}")
    for k, v in vals.items():
        print(f"{k}: {v}")

output = {
    "judge_model": JUDGE_MODEL,
    "embedding_model": EMBED_MODEL,
    "summary": summary,
}
with open(SUMMARY_JSON,"w",encoding="utf-8",) as f:
    json.dump(output, f, ensure_ascii=False, indent=2,)
combined_df = pd.concat(list(all_dfs.values()),ignore_index=True,)
combined_df.to_csv( DETAIL_CSV, index=False, encoding="utf-8-sig",)

print("\nDONE")
print(SUMMARY_JSON)
print(DETAIL_CSV)

/tmp/ipykernel_2727/2819043174.py:13: DeprecationWarning: Importing AnswerRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import AnswerRelevancy
  from ragas.metrics import (AnswerRelevancy,


Loading tokenizer...
Loading model...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/tmp/ipykernel_2727/2819043174.py:88: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  llm = LangchainLLMWrapper(hf_llm)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

Batch 48 | 40.2s | ETA 0p 39s


Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

Batch 49 | 30.2s | ETA 0p 0s

SUMMARY

RAG
answer_relevancy: 0.5162

GraphRAG
answer_relevancy: 0.4261

LLM_only
answer_relevancy: 0.3914

DONE
/content/drive/MyDrive/laborlaw/ragas/results_summary.json
/content/drive/MyDrive/laborlaw/ragas/results_detail.csv
